In [1]:
import numpy as np

**Download here:** https://www.kaggle.com/datasets/rtatman/glove-global-vectors-for-word-representation

In [2]:
GLOVE_FILE = "model/glove.6B.50d.txt/glove.6B.50d.txt"

In [3]:
with open(GLOVE_FILE, 'r', encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i < 3:  # print ... first lines
            print(line.strip())
            print('~'*100)
        else:
            break

the 0.418 0.24968 -0.41242 0.1217 0.34527 -0.044457 -0.49688 -0.17862 -0.00066023 -0.6566 0.27843 -0.14767 -0.55677 0.14658 -0.0095095 0.011658 0.10204 -0.12792 -0.8443 -0.12181 -0.016801 -0.33279 -0.1552 -0.23131 -0.19181 -1.8823 -0.76746 0.099051 -0.42125 -0.19526 4.0071 -0.18594 -0.52287 -0.31681 0.00059213 0.0074449 0.17778 -0.15897 0.012041 -0.054223 -0.29871 -0.15749 -0.34758 -0.045637 -0.44251 0.18785 0.0027849 -0.18411 -0.11514 -0.78581
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
, 0.013441 0.23682 -0.16899 0.40951 0.63812 0.47709 -0.42852 -0.55641 -0.364 -0.23938 0.13001 -0.063734 -0.39575 -0.48162 0.23291 0.090201 -0.13324 0.078639 -0.41634 -0.15428 0.10068 0.48891 0.31226 -0.1252 -0.037512 -1.5179 0.12612 -0.02442 -0.042961 -0.28351 3.5416 -0.11956 -0.014533 -0.1499 0.21864 -0.33412 -0.13872 0.31806 0.70358 0.44858 -0.080262 0.63003 0.32111 -0.46765 0.22786 0.36034 -0.37818 -0.56657 0.044691 0.30392
~~~~~~~~~~~~~~~~~~~

In [4]:
def read_GloVe_txt(file):
    with open(file, 'r', encoding='utf-8') as f:
        words = set()
        word2vec = {}
        for line in f:
            list = line.strip().split()
            word = list[0]
            vector = np.array(list[1:], dtype=np.float64)
            words.add(word)
            word2vec[word] = vector
    return words, word2vec

In [5]:
words, word2vec = read_GloVe_txt(GLOVE_FILE)

In [6]:
print("Number of words for embedding in GloVe:", len(words))
print("Embedding size of a word in GloVe:", word2vec['the'].shape)

Number of words for embedding in GloVe: 400000
Embedding size of a word in GloVe: (50,)


# Cosine Similarity

In [7]:
def cosine_similarity(u, v):
    dot_product = np.dot(u, v)
    magnitude_u = np.linalg.norm(u)
    magnitude_v = np.linalg.norm(v)

    if magnitude_u == 0 or magnitude_v == 0:
        return 0.0

    return dot_product / (magnitude_u * magnitude_v)

In [8]:
pairs = [
    ("good", "bad"),
    ("hot", "cold"),
    ("man", "woman"),
    ("happy", "sad"),
    ("cat", "dog"),
    ("king", "queen"),
    ("up", "down"),
    ("table", "banana")
]

for w1, w2 in pairs:
    if w1 in word2vec and w2 in word2vec:
        sim = cosine_similarity(word2vec[w1], word2vec[w2])
        print(f"{w1:>8s} – {w2:<8s} → cosine_similarity = {sim:.3f}")
    else:
        print(f"{w1}, {w2} không có trong word2vec.")

    good – bad      → cosine_similarity = 0.796
     hot – cold     → cosine_similarity = 0.801
     man – woman    → cosine_similarity = 0.886
   happy – sad      → cosine_similarity = 0.689
     cat – dog      → cosine_similarity = 0.922
    king – queen    → cosine_similarity = 0.784
      up – down     → cosine_similarity = 0.952
   table – banana   → cosine_similarity = 0.240


# Word Analogy Task

In [9]:
def find_analogy_word(word1, word2, word3, word2vec):
    w1, w2, w3 = word1.lower(), word2.lower(), word3.lower()
    if w1 not in word2vec or w2 not in word2vec or w3 not in word2vec:
        return None
        
    e1, e2, e3 = word2vec[w1], word2vec[w2], word2vec[w3]
    target_e = e2 - e1 + e3
    
    max_sim = -1
    best_word = None

    for word, e in word2vec.items():
        if word in [w1, w2, w3]:
            continue
        sim = cosine_similarity(target_e, e)
        if sim > max_sim:
            max_sim = sim
            best_word = word

    return best_word

In [10]:
examples = [
    ("man", "king", "woman"),    
    ("france", "paris", "italy"),   
    ("boy", "girl", "brother"),   
    ("walk", "walking", "swim"),    
    ("big", "bigger", "small"),
]

for a, b, c in examples:
    print(f"{a:>10s} : {b:<20s} :: {c:<10s} → {find_analogy_word(a, b, c, word2vec)}")

       man : king                 :: woman      → queen
    france : paris                :: italy      → rome
       boy : girl                 :: brother    → cousin
      walk : walking              :: swim       → swimming
       big : bigger               :: small      → larger


# Bias in Word Embedding

### Bias in Gender related to Name, Job, ...

In [11]:
gender_direction = word2vec['woman'] - word2vec['man']

In [12]:
print("List of names and their similarities with gender_direction:")

name_list = ['john', 'marie', 'sophie', 'ronaldo', 'priya', 'rahul', 'danielle', 'reza', 'katy', 'yasmin']

for w in name_list:
    if w in word2vec:
        sim = cosine_similarity(word2vec[w], gender_direction)
        print(f"{w:<10s}: {sim:+.3f}")

List of names and their similarities with gender_direction:
john      : -0.232
marie     : +0.316
sophie    : +0.319
ronaldo   : -0.312
priya     : +0.176
rahul     : -0.169
danielle  : +0.244
reza      : -0.079
katy      : +0.283
yasmin    : +0.233


In [13]:
print("Other words and their similarities with gender_direction:")

word_list = ['lipstick', 'guns', 'science', 'arts', 'literature', 'warrior', 'doctor', 'tree', 'receptionist', 'technology', 'fashion', 
             'teacher', 'engineer', 'pilot', 'computer', 'singer']

for w in word_list:
    if w in word2vec:
        sim = cosine_similarity(word2vec[w], gender_direction)
        print(f"{w:<15s}: {sim:+.3f}")

Other words and their similarities with gender_direction:
lipstick       : +0.277
guns           : -0.189
science        : -0.061
arts           : +0.008
literature     : +0.065
warrior        : -0.209
doctor         : +0.119
tree           : -0.071
receptionist   : +0.331
technology     : -0.132
fashion        : +0.036
teacher        : +0.179
engineer       : -0.080
pilot          : +0.001
computer       : -0.103
singer         : +0.185


# Debias

### Neutralize bias for non-gender specific words

In [14]:
def neutralize(word, bias_axis, word2vec):
    """
    To debias, e_debiased @ bias_axis = 0, means the angle between 2 vectors is 90 degrees
        e_debiased = e - e_bias
        e_bias = alpha * b  
        --> (e - alpha * b) @ b = 0  
        --> alpha = (e @ b) / (b @ b) 
        --> e_bias = [(e @ b) / L2_norm(b)**2] * b
    """
    
    e = word2vec[word]
    e_bias = (np.dot(e, bias_axis) / np.linalg.norm(bias_axis)**2) * bias_axis
    e_debiased = e - e_bias
    return e_debiased

In [15]:
print("=== Checking debias effect ===")
for w in word_list:
    if w in word2vec:
        before = cosine_similarity(word2vec[w], gender_direction)
        after = cosine_similarity(neutralize(w, gender_direction, word2vec), gender_direction)
        print(f"{w:<15s}: before = {before:+.3f}, after = {after:+.3f}")

=== Checking debias effect ===
lipstick       : before = +0.277, after = +0.000
guns           : before = -0.189, after = +0.000
science        : before = -0.061, after = +0.000
arts           : before = +0.008, after = +0.000
literature     : before = +0.065, after = +0.000
warrior        : before = -0.209, after = +0.000
doctor         : before = +0.119, after = +0.000
tree           : before = -0.071, after = -0.000
receptionist   : before = +0.331, after = +0.000
technology     : before = -0.132, after = +0.000
fashion        : before = +0.036, after = +0.000
teacher        : before = +0.179, after = +0.000
engineer       : before = -0.080, after = +0.000
pilot          : before = +0.001, after = +0.000
computer       : before = -0.103, after = -0.000
singer         : before = +0.185, after = +0.000


### Equalize for gender-specific words

In [16]:
def equalize(word_pair, bias_axis, word2vec):
    """
    To equalize, 2 word vectors must have the same angle (cosine) and be equidistant from the midpoint of the 2 vectors along the bias axis
    """

    w1, w2 = word_pair
    e1, e2 = word2vec[w1], word2vec[w2]

    e1_parallel = (np.dot(e1, bias_axis) / np.linalg.norm(bias_axis)**2) * bias_axis
    e2_parallel = (np.dot(e2, bias_axis) / np.linalg.norm(bias_axis)**2) * bias_axis

    # Compute midpoint
    mu = (e1 + e2) / 2.0
    # Split into 1 vector parallel to bias_axis, 1 vector orthogonal to bias_axis
    mu_parallel = (np.dot(mu, bias_axis) / np.linalg.norm(bias_axis)**2) * bias_axis
    mu_orth = mu - mu_parallel

    # Compute the direction from 2 vectors that are both parallel to bias_axis, to decide whether e_equalized sits on the left or right along bias axis
    # e_direction is the vector with length normalized to 1
    e1_direction = (e1_parallel - mu_parallel) / np.linalg.norm(e1_parallel - mu_parallel)
    e2_direction = (e2_parallel - mu_parallel) / np.linalg.norm(e2_parallel - mu_parallel)

    # Compute magnitude, e_equalized**2 (which is 1) = (mu_orth)**2 + (new_e_bias)**2
    # Manitude is the length of the vector new_e_bias
    magnitude = np.sqrt(np.abs(1 - np.sum(mu_orth**2)))
    
    # Equalization
    e1_equalized = mu_orth + magnitude * e1_direction
    e2_equalized = mu_orth + magnitude * e2_direction

    return e1_equalized, e2_equalized

In [17]:
gender_related_word_pairs = [
    ("man", "woman"),
    ("king", "queen"),
    ("actor", "actress"),
    ("husband", "wife"),
    ("prince", "princess")
]


for w1, w2 in gender_related_word_pairs:
    print(f"=== Pair: {w1} - {w2} ===")

    print("Before equalizing:")
    print(f"cosine_similarity('{w1}', gender_direction) =", cosine_similarity(word2vec[w1], gender_direction))
    print(f"cosine_similarity('{w2}', gender_direction) =", cosine_similarity(word2vec[w2], gender_direction))
    print()
    # Equalize
    e1, e2 = equalize((w1, w2), gender_direction, word2vec)
    print("After equalizing:")
    print(f"cosine_similarity('{w1}', gender_direction) =", cosine_similarity(e1, gender_direction))
    print(f"cosine_similarity('{w2}', gender_direction) =", cosine_similarity(e2, gender_direction))
    print('~'*100)
    print()

=== Pair: man - woman ===
Before equalizing:
cosine_similarity('man', gender_direction) = -0.1171109576533683
cosine_similarity('woman', gender_direction) = 0.3566661884627037

After equalizing:
cosine_similarity('man', gender_direction) = -0.7004364289309386
cosine_similarity('woman', gender_direction) = 0.7004364289309387
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

=== Pair: king - queen ===
Before equalizing:
cosine_similarity('king', gender_direction) = -0.18875684078397287
cosine_similarity('queen', gender_direction) = 0.2044893354882803

After equalizing:
cosine_similarity('king', gender_direction) = -0.6998245263520646
cosine_similarity('queen', gender_direction) = 0.6998245263520648
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

=== Pair: actor - actress ===
Before equalizing:
cosine_similarity('actor', gender_direction) = -0.08387555382505693
cosine_similarity('actr